# Robot Synthetic Data Generation Workshop

End-to-end pipeline: **Synthetic Data Generation → VLA Training → Simulation Evaluation**

| Step | Script | Description |
|------|--------|-------------|
| 0 | Setup | Install deps, check GPU, download scene assets |
| 1 | `01_gen_data.py` / `02_gen_data_custom_scene.py` | Generate Franka pick-cube trajectories |
| 2 | `02_train_vla.py` | Fine-tune SmolVLA on collected data |
| 3 | `03_eval.py` / `04_eval_custom_scene.py` | Closed-loop simulation evaluation |

---
## 0. Environment Setup

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
from IPython.display import display, Image, HTML, Video
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
matplotlib.rcParams['figure.dpi'] = 120

os.environ['HF_HUB_OFFLINE'] = '1'

WORKSHOP_DIR = Path(os.getcwd())
SCRIPTS_DIR = WORKSHOP_DIR / 'scripts'
OUTPUT_DIR = Path('/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Workshop dir: {WORKSHOP_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Python:       {sys.executable}')

### 0.1 GPU & ROCm Check

In [ ]:
import torch

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU[{i}]: {props.name}  VRAM: {vram_gb:.1f} GB')
else:
    print('WARNING: No GPU detected.')

!rocm-smi --showuse 2>/dev/null | head -20 || echo 'rocm-smi not available'

### 0.2 Install Dependencies

In [ ]:
!pip install -q lerobot genesis-world transformers accelerate safetensors matplotlib Pillow 2>&1 | tail -5

In [ ]:
import lerobot
print(f'lerobot: {lerobot.__version__}')

try:
    import genesis as gs
    print(f'genesis: {gs.__version__}')
except Exception as e:
    print(f'genesis import: {e}')

### 0.3 Download Kitchen Scene Assets (for custom-scene steps)

In [ ]:
!python {SCRIPTS_DIR}/00_download_kitchen.py --mesh-only

In [ ]:
asset_dir = WORKSHOP_DIR / 'assets' / 'rustic_kitchen'
if asset_dir.exists():
    for f in sorted(asset_dir.iterdir()):
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:40s}  {size_mb:>8.1f} MB')
else:
    print(f'Asset dir not found: {asset_dir}')

---
## 1. Synthetic Data Generation

Generate Franka 7-DOF pick-cube expert trajectories using IK planning in Genesis.

- **Robot**: Franka Panda (9 DOF: 7 arm + 2 finger)
- **Trajectory**: HOME → hover → descend → grasp → lift
- **Cameras**: up + side (640×480)

### 1a. Default Flat Scene

In [ ]:
N_EPISODES_FLAT = 10
REPO_ID_FLAT = 'local/workshop-flat'

!python {SCRIPTS_DIR}/01_gen_data.py \
    --n-episodes {N_EPISODES_FLAT} \
    --repo-id {REPO_ID_FLAT} \
    --save {OUTPUT_DIR} \
    --seed 42 \
    --no-videos \
    --task 'Pick up the red cube.'

### 1b. Custom Kitchen Scene

In [ ]:
N_EPISODES_KITCHEN = 10
REPO_ID_KITCHEN = 'local/workshop-kitchen'

!python {SCRIPTS_DIR}/02_gen_data_custom_scene.py \
    --n-episodes {N_EPISODES_KITCHEN} \
    --repo-id {REPO_ID_KITCHEN} \
    --save {OUTPUT_DIR} \
    --seed 42 \
    --no-videos \
    --task 'Pick up the red cube.'

### 1c. Visualize Generated Data

In [ ]:
try:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
except ImportError:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds_root = Path.home() / '.cache' / 'huggingface' / 'lerobot' / REPO_ID_FLAT
ds = LeRobotDataset(REPO_ID_FLAT, root=ds_root)
print(f'Dataset: {REPO_ID_FLAT}  (root: {ds_root})')
print(f'  Episodes:     {ds.num_episodes}')
print(f'  Total frames: {len(ds)}')
print(f'  FPS:          {ds.fps}')
print(f'  Features:     {list(ds.features.keys())}')

sample = ds[0]
print(f'\nSample keys: {list(sample.keys())}')
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: shape={v.shape} dtype={v.dtype}')
    elif isinstance(v, str):
        print(f'  {k}: "{v}"')
    else:
        print(f'  {k}: {type(v).__name__} = {v}')

In [ ]:
# Visualize camera images from a sample episode
from PIL import Image as PILImage

def get_episode_range(dataset, ep_idx):
    if hasattr(dataset, 'episode_data_index'):
        start = dataset.episode_data_index['from'][ep_idx].item()
        end = dataset.episode_data_index['to'][ep_idx].item()
    else:
        fpe = len(dataset) // dataset.num_episodes
        start, end = ep_idx * fpe, (ep_idx + 1) * fpe
    return start, end

def show_episode_frames(dataset, ep_idx=0, n_frames=6):
    start, end = get_episode_range(dataset, ep_idx)
    indices = np.linspace(start, end - 1, n_frames, dtype=int)
    img_keys = [k for k in dataset.features if 'images' in k]
    if not img_keys:
        print('No image features found')
        return

    fig, axes = plt.subplots(len(img_keys), n_frames,
                             figsize=(3 * n_frames, 3 * len(img_keys)))
    if len(img_keys) == 1:
        axes = axes[np.newaxis, :]

    for row, key in enumerate(img_keys):
        for col, idx in enumerate(indices):
            sample = dataset[int(idx)]
            img = sample[key]
            if hasattr(img, 'numpy'):
                img = img.numpy()
            if img.ndim == 3 and img.shape[0] in (1, 3):
                img = np.transpose(img, (1, 2, 0))
            if img.dtype != np.uint8:
                if img.max() <= 1.0:
                    img = (img * 255).astype(np.uint8)
                else:
                    img = img.astype(np.uint8)
            axes[row, col].imshow(img)
            axes[row, col].set_title(f'frame {idx - start}', fontsize=8)
            axes[row, col].axis('off')
        axes[row, 0].set_ylabel(key.split('.')[-1], fontsize=10)

    fig.suptitle(f'Episode {ep_idx} Camera Views', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'ep{ep_idx}_camera_views.png'), bbox_inches='tight')
    plt.show()
    print(f'Saved: {OUTPUT_DIR / f"ep{ep_idx}_camera_views.png"}')

show_episode_frames(ds, ep_idx=0)

In [ ]:
# Plot joint trajectories for episode 0
def plot_joint_trajectory(dataset, ep_idx=0):
    start, end = get_episode_range(dataset, ep_idx)
    states, actions = [], []
    for i in range(start, end):
        s = dataset[i]
        states.append(s['observation.state'].numpy())
        actions.append(s['action'].numpy())

    states = np.array(states)
    actions = np.array(actions)
    joint_names = ['j1','j2','j3','j4','j5','j6','j7','f1','f2']

    fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
    for i, ax in enumerate(axes.flat):
        if i >= states.shape[1]:
            ax.set_visible(False)
            continue
        ax.plot(states[:, i], label='state', linewidth=1)
        ax.plot(actions[:, i], label='action', linewidth=1, alpha=0.7)
        name = joint_names[i] if i < len(joint_names) else f'dim{i}'
        ax.set_title(name, fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f'Episode {ep_idx} Joint Trajectories (state vs action)', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'ep{ep_idx}_joint_trajectory.png'), bbox_inches='tight')
    plt.show()

plot_joint_trajectory(ds, ep_idx=0)

In [ ]:
# Cube XY scatter: show all episode spawn positions + success/fail
def plot_cube_scatter(gen_dir):
    labels_path = gen_dir / 'episode_labels.json'
    summary_path = gen_dir / 'gen_summary.json'
    if not labels_path.exists():
        print(f'Labels not found: {labels_path}')
        return
    labels = json.loads(labels_path.read_text())
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

    fig, ax = plt.subplots(figsize=(6, 5))
    for ep in labels:
        xy = ep.get('cube_xy') or ep.get('cube_world_xy') or ep.get('cube_local_dxdy')
        if xy is None:
            continue
        color = '#2ecc71' if ep['success'] else '#e74c3c'
        marker = 'o' if ep['success'] else 'x'
        ax.scatter(xy[0], xy[1], c=color, marker=marker, s=60, edgecolors='k', linewidths=0.5)

    ax.set_xlabel('X'); ax.set_ylabel('Y')
    sr = summary.get('success_rate', 0)
    ax.set_title(f'Cube Spawn Positions \u2014 SR: {sr:.0%}')
    ax.grid(True, alpha=0.3); ax.set_aspect('equal')
    plt.tight_layout()
    out = gen_dir / 'cube_xy_scatter.png'
    fig.savefig(str(out), bbox_inches='tight')
    plt.show()

print('=== Flat Scene ===')
plot_cube_scatter(OUTPUT_DIR / 'franka_gen_pick')
print('\n=== Kitchen Scene ===')
plot_cube_scatter(OUTPUT_DIR / 'custom_scene_gen')

---
## 2. SmolVLA Post-Training

Fine-tune `lerobot/smolvla_base` (~450M params) on the generated dataset.
- Freeze vision encoder, train expert layers + state projection
- chunk_size=50, AdamW optimizer

> **Note**: SmolVLA training requires specific lerobot versions for full compatibility.

In [ ]:
N_TRAIN_STEPS = 200
BATCH_SIZE = 4
TRAIN_REPO = REPO_ID_FLAT
SAVE_DIR = OUTPUT_DIR / 'outputs' / 'workshop_smolvla'

!python {SCRIPTS_DIR}/02_train_vla.py \
    --dataset-id {TRAIN_REPO} \
    --n-steps {N_TRAIN_STEPS} \
    --batch-size {BATCH_SIZE} \
    --num-workers 0 \
    --save-dir {SAVE_DIR}

### 2.1 Training Loss Curve

In [ ]:
def plot_loss_curve(save_dir):
    metrics_path = save_dir / 'train_metrics.json'
    summary_path = save_dir / 'train_summary.json'
    if not metrics_path.exists():
        print(f'Metrics not found: {metrics_path} (training may not have completed)')
        return

    metrics = json.loads(metrics_path.read_text())
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

    steps = [m['step'] for m in metrics]
    losses = [m['loss'] for m in metrics]
    grad_norms = [m['grad_norm'] for m in metrics]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps, losses, linewidth=0.8, alpha=0.5, label='per-step')
    window = max(1, len(losses) // 20)
    if len(losses) > window:
        smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
        ax1.plot(steps[window-1:], smoothed, linewidth=2, color='#e74c3c', label=f'smooth(w={window})')
    ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(steps, grad_norms, linewidth=0.8, alpha=0.6, color='#3498db')
    ax2.set_xlabel('Step'); ax2.set_ylabel('Grad Norm')
    ax2.set_title('Gradient Norm'); ax2.grid(True, alpha=0.3)

    info = ''
    if summary:
        info = (f"loss: {summary.get('loss_start','?'):.4f} -> {summary.get('loss_end','?'):.4f}  "
                f"time: {summary.get('total_time_s',0):.0f}s  VRAM: {summary.get('peak_vram_mb',0):.0f}MB")
    fig.suptitle(f'SmolVLA Training  |  {info}', fontsize=10)
    plt.tight_layout()
    fig.savefig(str(save_dir / 'loss_curve.png'), bbox_inches='tight')
    plt.show()

plot_loss_curve(SAVE_DIR)

---
## 3. Simulation Evaluation

Closed-loop: render cameras → SmolVLA inference → PD control → step physics.

> **Note**: Eval requires a trained checkpoint from Step 2.

In [ ]:
CHECKPOINT = SAVE_DIR / 'final'
EVAL_UNSEEN_DIR = OUTPUT_DIR / 'eval_unseen'

if CHECKPOINT.exists():
    !python {SCRIPTS_DIR}/03_eval.py \
        --policy-type smolvla \
        --checkpoint {CHECKPOINT} \
        --dataset-id {TRAIN_REPO} \
        --n-episodes 5 --max-steps 150 --seed 99 \
        --record-video \
        --save {EVAL_UNSEEN_DIR}
else:
    print(f'No checkpoint at {CHECKPOINT} - skipping eval (training may not have completed)')

---
## 4. Summary & Artifacts

In [ ]:
print('=== Generated Artifacts ===')
pngs = sorted(OUTPUT_DIR.rglob('*.png'))
mp4s = sorted(OUTPUT_DIR.rglob('*.mp4'))
jsons = sorted(OUTPUT_DIR.rglob('*.json'))

print(f'\nPNG images ({len(pngs)}):')
for p in pngs:
    print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1024:.0f} KB)')

print(f'\nMP4 videos ({len(mp4s)}):')
for p in mp4s:
    print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1024**2:.1f} MB)')

print(f'\nJSON summaries ({len(jsons)}):')
for p in jsons:
    print(f'  {p.relative_to(OUTPUT_DIR)}')

In [ ]:
# Display all generated PNGs inline
for p in pngs:
    print(f'\n--- {p.relative_to(OUTPUT_DIR)} ---')
    display(Image(filename=str(p), width=700))

In [ ]:
# Show data generation summaries
for gen_name in ['franka_gen_pick', 'custom_scene_gen']:
    sp = OUTPUT_DIR / gen_name / 'gen_summary.json'
    if sp.exists():
        summary = json.loads(sp.read_text())
        print(f'\n=== {gen_name} ===')
        for k, v in summary.items():
            if k not in ('success_episode_ids', 'failure_episode_ids'):
                print(f'  {k}: {v}')
        print(f'  success_rate: {summary.get("success_rate", "N/A")}')

In [ ]:
print('\nWorkshop pipeline complete!')
print(f'All outputs saved to: {OUTPUT_DIR}')